In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import sys

#math and array operations
import numpy as np
import math

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#data classes
import xarray as xr
import h5py
import pickle 

#loading bar
from tqdm import tqdm

#dates
from datetime import datetime

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "MPAS_Model_Data", "Snapshots")
dataType = "ConvectiveCores"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
#Setup

Region = "TRACER"; Case = "WET"; spinup_hours = "0"
Region = "TRACER"; Case = "DIURNAL"; spinup_hours = "-5"
Region = "PRECIP"; Case = "WET"; spinup_hours = "12"
Region = "PRECIP"; Case = "DIURNAL"; spinup_hours = "12"

Region = "Hawaii"; Case = "WET"; spinup_hours = "12"
Region = "Hawaii"; Case = "TRADES"; spinup_hours = "24" 

In [ ]:
#Load Model Directory Class
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData_NSSL = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

RunType = (Region,Case,"TEMPO",spinup_hours)
ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_DataSaving import DataSaving_Class

In [ ]:
#Importing PlottingModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_PlottingModelData import RadarPlotting_Class

In [ ]:
#Importing Radar Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","Observation_Data"))
from CLASSES_RadarDataLoading import RadarData_MRMS_Class, RadarObservationMask_Class

sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_RadarDataPlotting import RadarPlotting_Class

In [ ]:
#IMPORT FUNCTIONS
# --- Add your Functions folder to sys.path ---
import sys
path = os.path.join(DirectoryManager.mainCodeDirectory, 'Functions_2.0')
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "AreaAverageFunctions",
    "ComputationFunctions",
    "DataFunctions",
    "DerivativeFunctions",
    "PlottingFunctions",
    "StatisticalFunctions",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [ ]:
###############
#JOB ARRAY SETUP

In [ ]:
#Importing PlottingModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_JobArray import JobArray_Class

In [ ]:
####################################
#CALCULATION FUNCTIONS

In [ ]:
def GetMean(variableSubset, mode="xy"):
    #(1/A) ∫ φ dA
    # dA = R² cos(lat) dlat dlon → weight = cos(lat)

    weights = np.cos(np.deg2rad(variableSubset.latitude))

    if mode == "xy":
        dims = ("latitude", "longitude")
    elif mode == "y":
        dims = ("latitude",)
    elif mode == "x":
        dims = ("longitude",)
    else:
        raise ValueError("mode must be 'xy' or 'y'")

    variableMean = variableSubset.weighted(weights).mean(
        dim=dims,
        skipna=True
    )

    return variableMean

def MeanDBZ(variableSubset, mode="xy"):
    # Convert dBZ → linear Z
    variableSubset_power = 10 ** (variableSubset / 10.0)

    # Take weighted mean
    variableMean = GetMean(variableSubset_power, mode=mode)

    # Convert back to dBZ
    variableMean = 10.0 * np.log10(variableMean)

    return variableMean

In [ ]:
####################################
#LOADING FUNCTIONS

In [ ]:
def GetData(ModelData, varName, t, zTarget=None):
        
    #Loading Data
    [dataSubset,dataSubset_diag,dataSubset_static, lat,lon,zGrid_f,zGrid_c, _, _] = DataOperator_Class.GetData_Subset(ModelData, t)
        
    #Subsetting Data
    variableSubset= GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName)
    
    #Interpolating Z levels #*INTERPOLATION
    #################################
    if any(dim.startswith("nVertLevels") for dim in variableSubset.dims):
        if zTarget is None:
            [zTarget_f, zTarget_c] = ModelData.GetZTarget(zGrid_f, zGrid_c)
            zTarget = "loaded"
        variableSubset = ModelData.InterpolateVertical(variableSubset,zGrid_f,zGrid_c,zTarget_f,zTarget_c)
    #################################            
    
    if varName in ['refl10cm','refl10cm_1km']:
        variableSubset = variableSubset.where(variableSubset > 0)

    return variableSubset
    
# def GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName):
#     """
#     Retrieves a variable subset from the given model data.
#     If varName contains a '+', returns the sum of the two variables.
#     """
#     if '+' in varName:
#         var1, var2 = varName.split('+')
#         var1 = var1.strip()
#         var2 = var2.strip()

#         subset1 = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
#                                                       dataSubset_diag, dataSubset_static, var1)
#         subset2 = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
#                                                       dataSubset_diag, dataSubset_static, var2)
#         variableSubset = subset1 + subset2
#     else:
#         variableSubset = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
#                                                              dataSubset_diag, dataSubset_static, varName)

#     return variableSubset

def GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName):
    """
    Retrieves a variable subset from the given model data.
    Supports summing multiple variables if varName contains '+' 
    (e.g., 'qc+qi+qg+qs').
    """

    if '+' in varName:
        varList = [v.strip() for v in varName.split('+')]

        variablesum = None
        for var in varList:
            subset = DataOperator_Class.GetData_Variable(
                ModelData,
                dataSubset,
                dataSubset_diag,
                dataSubset_static,
                var
            )

            if variablesum is None:
                variablesum = subset
            else:
                variablesum = variablesum + subset

        variableSubset = variablesum

    else:
        variableSubset = DataOperator_Class.GetData_Variable(
            ModelData,
            dataSubset,
            dataSubset_diag,
            dataSubset_static,
            varName
        )

    return variableSubset

In [ ]:
def GetAllNeededVariables(varName, reflThreshold=15):
    
    # Other Variables
    varData_NSSL = GetData(ModelData_NSSL, varName.replace("'",""),t)
    varData_TEMPO = GetData(ModelData_TEMPO, varName.replace("'",""),t)
    
    zdim = GetVerticalDim(varData_NSSL)
    MeanFunction = GetMean if varName != "refl10cm" else MeanDBZ
    
    if "'" in varName:
        varData_NSSL=varData_NSSL-MeanFunction(varData_NSSL) #for perturbations
        varData_TEMPO=varData_TEMPO-MeanFunction(varData_TEMPO) #for perturbations
    
    #reflectivity threshold
    reflData_NSSL = GetData(ModelData_NSSL, "refl10cm",t)
    reflData_TEMPO = GetData(ModelData_TEMPO, "refl10cm",t)
    if "nVertLevelsP1" in varData_NSSL.coords:
        reflData_NSSL=reflData_NSSL.interp(nVertLevels=varData_NSSL.nVertLevelsP1)
        reflData_TEMPO=reflData_TEMPO.interp(nVertLevels=varData_NSSL.nVertLevelsP1)
    reflData_NSSL = reflData_NSSL.sel({zdim: 3000}, method='nearest') #we only want to see above the storm objects at 3km
    reflData_TEMPO = reflData_TEMPO.sel({zdim: 3000}, method='nearest')
    varData_NSSL_threshold = varData_NSSL.where(reflData_NSSL>reflThreshold)
    varData_TEMPO_threshold = varData_TEMPO.where(reflData_TEMPO>reflThreshold)
    
    subsetData_NSSL = varData_NSSL_threshold.sel({
        zdim: slice(0,16000),
        "longitude": slice(lon_min, lon_max),
        "latitude": slice(lat_min, lat_max)
    })    
    subsetData_TEMPO = varData_TEMPO_threshold.sel({
        zdim: slice(0,16000),
        "longitude": slice(lon_min, lon_max),
        "latitude": slice(lat_min, lat_max)
    })
    
    #mean
    varMean_NSSL = MeanFunction(subsetData_NSSL,mode="y")
    varMean_TEMPO = MeanFunction(subsetData_TEMPO,mode="y")

    varProfile_NSSL = MeanFunction(subsetData_NSSL,mode="xy")
    varProfile_TEMPO = MeanFunction(subsetData_TEMPO,mode="xy")

    return varData_NSSL,varData_TEMPO, subsetData_NSSL,subsetData_TEMPO, varMean_NSSL,varMean_TEMPO, varProfile_NSSL,varProfile_TEMPO

In [ ]:
def LoadOrSave(varName):

    filePath = DataOperator_Class.GetOutputFilePath(
        ModelData_NSSL,
        DirectoryManager,
        outputDirectory,
        fileName=f"outputDictionary_{varName}.nc"
    )

    # ---------------------------------
    # LOAD
    # ---------------------------------
    if os.path.exists(filePath):
        print(f"Loading {filePath}")

        ds = xr.open_dataset(filePath)

        varData_NSSL  = ds["varData_NSSL"]
        varData_TEMPO = ds["varData_TEMPO"]
        subsetData_NSSL  = ds["subsetData_NSSL"]
        subsetData_TEMPO = ds["subsetData_TEMPO"]
        varMean_NSSL     = ds["varMean_NSSL"]
        varMean_TEMPO    = ds["varMean_TEMPO"]
        varProfile_NSSL  = ds["varProfile_NSSL"]
        varProfile_TEMPO = ds["varProfile_TEMPO"]

    # ---------------------------------
    # RUN + SAVE
    # ---------------------------------
    else:
        print(f"Running GetAllNeededVariables for {varName}")

        varData_NSSL, varData_TEMPO, \
        subsetData_NSSL, subsetData_TEMPO, \
        varMean_NSSL, varMean_TEMPO, \
        varProfile_NSSL, varProfile_TEMPO = GetAllNeededVariables(varName=varName)

        os.makedirs(os.path.dirname(filePath), exist_ok=True)

        ds = xr.Dataset({
            "varData_NSSL": varData_NSSL,
            "varData_TEMPO": varData_TEMPO,
            "subsetData_NSSL": subsetData_NSSL,
            "subsetData_TEMPO": subsetData_TEMPO,
            "varMean_NSSL": varMean_NSSL,
            "varMean_TEMPO": varMean_TEMPO,
            "varProfile_NSSL": varProfile_NSSL,
            "varProfile_TEMPO": varProfile_TEMPO,
        })

        ds.to_netcdf(filePath)
        print(f"Saved to {filePath}")

    return (
        varData_NSSL,
        varData_TEMPO,
        subsetData_NSSL,
        subsetData_TEMPO,
        varMean_NSSL,
        varMean_TEMPO,
        varProfile_NSSL,
        varProfile_TEMPO
    )

In [ ]:
####################################
#LOADING DICTIONARIES

In [ ]:
timeStringDictionary = {
    "TRACER": {
        "WET": {
            0: f"2022-07-01_18.00.00",
        },
        "DIURNAL": {
            -5: f"2022-06-22_21.00.00",
        },
    },
    "PRECIP": {
        "WET": {
            12: f"2022-06-07_03.00.00",
        },
        "DIURNAL": {
            12: f"2022-07-16_05.00.00",
        },
    },
    "Hawaii": {
        "WET": {
            12: f"2021-12-06_23.00.00",
        },
        "TRADES": {
            24: f"2022-08-08_00.45.00",#f"2022-08-08_01.00.00",
        },
    },
}

selectedTimeString = timeStringDictionary[ModelData_NSSL.region][ModelData_NSSL.case][int(ModelData_NSSL.spinup_hours)]
t = ModelData_NSSL.timeStrings.index(selectedTimeString)

In [ ]:
rectangleBoundsDictionary = {
    "TRACER": {
        "WET": {
            0: [-94.5,-92, 28.5,29.9], #[lon_min,lon_max, lat_min,lat_max]
        },
        "DIURNAL": {
            -5: [-96,-92, 29.85,32.75],
        },
    },
    "PRECIP": {
        "WET": {
            12: [119.5,121.25, 22.25,24.25],
        },
        "DIURNAL": {
            12: [120.2,122.25, 22.4,25],
        },
    },
    "Hawaii": {
        "WET": {
            12: [-158,-156.75, 18.75,22],
        },
        "TRADES": {
            24: [-160.5,-159.25, 21.5,22.5],
        },
    },
}

[lon_min,lon_max, lat_min,lat_max] = rectangleBoundsDictionary[ModelData_NSSL.region][ModelData_NSSL.case][int(ModelData_NSSL.spinup_hours)]

In [ ]:
climLevelsDictionary = {"refl10cm": np.linspace(15,50,50),
                        "qc+qi+qg": np.linspace(0,1,50),
                        "w": np.linspace(-2,2,50),
                        "qr": np.linspace(0,1,50),
                        "relhum": np.linspace(50,105,50)
                       }

cmapDictionary = {"refl10cm": "turbo",
                  "qc+qi+qg": "turbo",
                  "w": "RdBu_r",
                  "qr": "turbo",
                  "relhum": "turbo"
                 }


In [ ]:
def MakeVariablesDictionary(varName,
                            varData_NSSL, varData_TEMPO,
                            subsetData_NSSL, subsetData_TEMPO,
                            varMean_NSSL, varMean_TEMPO,
                            varProfile_NSSL, varProfile_TEMPO,
                            variablesDictionary=None):

    if variablesDictionary is None:
        variablesDictionary = {}
    variablesDictionary[varName] = {
        "varData_NSSL": varData_NSSL,
        "varData_TEMPO": varData_TEMPO,
        "subsetData_NSSL": subsetData_NSSL,
        "subsetData_TEMPO": subsetData_TEMPO,
        "varMean_NSSL": varMean_NSSL,
        "varMean_TEMPO": varMean_TEMPO,
        "varProfile_NSSL": varProfile_NSSL,
        "varProfile_TEMPO": varProfile_TEMPO
    }

    return variablesDictionary




In [ ]:
####################################
#PLOTTING FUNCTIONS

In [ ]:
def PlotRectangle(ax,
                  lon_min, lon_max,
                  lat_min, lat_max):
    
    rect = Rectangle(
        (lon_min, lat_min),
        lon_max - lon_min,
        lat_max - lat_min,
        linewidth=2,
        edgecolor='red',
        facecolor='none'
    )
    
    ax.add_patch(rect)
from matplotlib.patches import Rectangle

In [ ]:
# def PlotRectangleFromCenterLine(ax,
#                                 lon0, lat0,
#                                 lon1, lat1,
#                                 width,
#                                 edgecolor='red',
#                                 linewidth=2):

#     # direction vector
#     dx = lon1 - lon0
#     dy = lat1 - lat0
#     L = np.sqrt(dx**2 + dy**2)

#     ux = dx / L
#     uy = dy / L

#     # perpendicular vector
#     px = -uy
#     py = ux

#     # half width
#     w = width / 2

#     # rectangle corners
#     p1 = (lon0 + px*w, lat0 + py*w)
#     p2 = (lon1 + px*w, lat1 + py*w)
#     p3 = (lon1 - px*w, lat1 - py*w)
#     p4 = (lon0 - px*w, lat0 - py*w)

#     rect = Polygon([p1, p2, p3, p4],
#                    closed=True,
#                    edgecolor=edgecolor,
#                    facecolor='none',
#                    linewidth=linewidth)

#     ax.add_patch(rect)
# from matplotlib.patches import Polygon

In [ ]:
def GetVerticalDim(data):
    if "nVertLevels" in data.dims:
        return "nVertLevels"
    elif "nVertLevelsP1" in data.dims:
        return "nVertLevelsP1"
    else:
        raise ValueError("No vertical dimension found")

In [ ]:
def PlotFullReflectivity(mpType,
                         ax=None):
    
    varName = "refl10cm"
    
    #Make Plot
    if ax is None:
        fig, ax = plt.subplots()
    p = variablesDictionary[varName][f"varData_{mpType}"].sel(nVertLevels=3000, method='nearest').plot(
        ax=ax, cmap='turbo',add_labels=False,add_colorbar=False
    )
    # p.colorbar.set_label('refl10cm')
    ax.set_title(varName, loc='center', fontsize=12, pad=3)
    
    #Draw Rectangle
    PlotRectangle(ax,
                  lon_min, lon_max,
                  lat_min, lat_max)

    return p

In [ ]:
def PlotHorizontal(subsetData, varName,
                   ax=None, cmap=None, levels=None):

    scaleFactor = 1000 if 'q' in varName else 1
    subsetData_plot = subsetData * scaleFactor

    if ax is None:
        fig, ax = plt.subplots()

    zdim = GetVerticalDim(subsetData)

    subsetData_plot = subsetData_plot.sel({
        zdim: slice(0,16000),
        "longitude": slice(lon_min, lon_max),
        "latitude": slice(lat_min, lat_max)
    })

    if cmap is None:
        cmap = 'turbo'

    p = subsetData_plot.sel({zdim:3000}, method='nearest').plot(
        ax=ax,
        cmap=cmap,
        levels=levels,
        add_colorbar=False,
        add_labels=False,
        extend="neither"
    )

    ax.set_title(varName, loc='center', fontsize=12, pad=3)

    return p

def PlotVertical(varMean, varName,
                 ax=None, cmap=None, levels=None):

    scaleFactor = 1000 if 'q' in varName else 1
    varMean_plot = varMean * scaleFactor

    zdim = GetVerticalDim(varMean)
    varMean_plot = varMean_plot.assign_coords({zdim: varMean_plot[zdim] / 1000})
    if ax is None:
        fig, ax = plt.subplots()

    varMean_plot = varMean_plot.sel({
        "longitude": slice(lon_min, lon_max)
    })

    if cmap is None:
        cmap = 'turbo'

    p = varMean_plot.plot(
        ax=ax,
        cmap=cmap,
        levels=levels,
        add_colorbar=False,
        extend="neither"
    )
    ax.set_ylim(0,16);
    ax.set_yticks(np.arange(0, 17, 2))
    ax.set_title(varName, loc='center', fontsize=12, pad=3)

    return p

def PlotHorizontalVertical_V1(subsetData, varMean, varName,
                           cmap='turbo', levels=None):

    fig = plt.figure(figsize=(12,4))

    outer = gridspec.GridSpec(1, 2, width_ratios=[20,1], wspace=0.05)

    inner = gridspec.GridSpecFromSubplotSpec(
        1, 2, subplot_spec=outer[0], wspace=0.25
    )

    ax0 = fig.add_subplot(inner[0])
    ax1 = fig.add_subplot(inner[1])
    cax = fig.add_subplot(outer[1])

    p = PlotHorizontal(subsetData, varName, ax0, cmap, levels)
    PlotVertical(varMean, varName, ax1, cmap, levels)

    fig.colorbar(p, cax=cax, label=varName)

    ax0.set_title("Horizontal Zoom-In")
    ax1.set_title("Average")

    return fig


def PlotHorizontalVertical_V2(subsetData, varMean, varName,
                           cmap='turbo', levels=None):

    fig = plt.figure(figsize=(12,4))

    outer = gridspec.GridSpec(1, 2, width_ratios=[20,1], wspace=0.05)

    inner = gridspec.GridSpecFromSubplotSpec(
        1, 2, subplot_spec=outer[0], wspace=0.25
    )

    ax0 = fig.add_subplot(inner[0])
    ax1 = fig.add_subplot(inner[1])
    cax = fig.add_subplot(outer[1])

    p = PlotHorizontal(subsetData, varName, ax0, cmap, levels)
    PlotVertical(varMean, varName, ax1, cmap, levels)

    fig.colorbar(p, cax=cax, label=varName)

    ax0.set_title("Horizontal Zoom-In")
    ax1.set_title("Average")

    return fig
# #EXAMPLE
# mpType = "NSSL"
# fig = PlotHorizontalVertical_V1(variablesDictionary[varName][f"subsetData_{mpType}"],variablesDictionary[varName][f"varMean_{mpType}"],varName=varName,
#                        cmap=cmapDictionary[varName],levels=climLevelsDictionary[varName])
# SaveFigure(ModelData_NSSL, fig, plotType=f"HorizontalVertical_NSSL_{varName}")

def PlotHorizontalVertical_V2(subsetData_NSSL, varMean_NSSL, 
                               subsetData_TEMPO, varMean_TEMPO, 
                               varName, cmap='turbo', levels=None):

    # Increased figure height to 8 to accommodate the second row
    fig = plt.figure(figsize=(12, 8))

    # Outer grid now has 2 rows, 2 columns (Plots column and Colorbar column)
    outer = gridspec.GridSpec(2, 2, width_ratios=[20, 1], wspace=0.05, hspace=0.3)

    # --- ROW 1: NSSL ---
    inner_nssl = gridspec.GridSpecFromSubplotSpec(
        1, 2, subplot_spec=outer[0, 0], wspace=0.25
    )
    ax0_nssl = fig.add_subplot(inner_nssl[0])
    ax1_nssl = fig.add_subplot(inner_nssl[1])

    # --- ROW 2: TEMPO ---
    inner_tempo = gridspec.GridSpecFromSubplotSpec(
        1, 2, subplot_spec=outer[1, 0], wspace=0.25
    )
    ax0_tempo = fig.add_subplot(inner_tempo[0])
    ax1_tempo = fig.add_subplot(inner_tempo[1])

    # Shared Colorbar Axis (Spans both rows on the right)
    cax = fig.add_subplot(outer[:, 1])

    # Plot NSSL Row
    p = PlotHorizontal(subsetData_NSSL, varName, ax0_nssl, cmap, levels)
    PlotVertical(varMean_NSSL, varName, ax1_nssl, cmap, levels)
    
    # Plot TEMPO Row
    PlotHorizontal(subsetData_TEMPO, varName, ax0_tempo, cmap, levels)
    PlotVertical(varMean_TEMPO, varName, ax1_tempo, cmap, levels)

    # Add the shared colorbar
    cb = fig.colorbar(p, cax=cax, label=varName)
    
    # Apply your 1-decimal formatting fix
    import matplotlib.ticker as ticker
    cb.ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))

    # Setting Titles
    ax0_nssl.set_title("NSSL: Horizontal Zoom-In")
    ax1_nssl.set_title("NSSL: Average")
    ax0_tempo.set_title("TEMPO: Horizontal Zoom-In")
    ax1_tempo.set_title("TEMPO: Average")

    return fig

In [ ]:
def PlotProfiles(varProfile_NSSL, varProfile_TEMPO, varName, ax=None):
    if ax is None:
        fig, ax = plt.subplots()
    
    # 1. Scaling logic
    scaleFactor = 1000 if 'q' in varName else 1
    
    # 2. Get vertical dimension and convert meters to km
    zdim = GetVerticalDim(varProfile_NSSL)
    
    # Create plotting data with km coordinates
    # We do this once to keep the code efficient
    vn_plot = (varProfile_NSSL * scaleFactor).assign_coords({zdim: varProfile_NSSL[zdim] / 1000})
    vt_plot = (varProfile_TEMPO * scaleFactor).assign_coords({zdim: varProfile_TEMPO[zdim] / 1000})

    # 3. Plotting lines
    ax.plot(vn_plot, vn_plot[zdim], color='blue', label='NSSL', linewidth=2)
    ax.plot(vt_plot, vt_plot[zdim], color='green', label='TEMPO', linewidth=2)

    # 4. Range and Ticks
    ax.set_ylim(0, 16)
    ax.set_yticks(np.arange(0, 17, 2))
    
    # 5. FONT SETTINGS (The Fix)
    # Ticks (the numbers)
    ax.tick_params(axis='both', labelsize=fontSettings["tickFont"])
    
    # Axis Labels (the text)
    ax.set_xlabel(varName, fontsize=fontSettings["labelFont"]+6)
    ax.set_ylabel("Altitude (km)", fontsize=fontSettings["labelFont"]+6)
    
    # Grid
    ax.grid(True, linestyle='--', alpha=0.7)
    
    return ax

In [ ]:
def GetOutputFile(ModelData, outputPlottingDirectory):
    outputSubDirectory = f"{ModelData.region}_{ModelData.case}_{ModelData.spinup_hours}hrs"
    
    outputFilePath = os.path.join(
        outputPlottingDirectory,
        outputSubDirectory)
    os.makedirs(outputFilePath, exist_ok=True)
    return outputFilePath


def SaveFigure(ModelData, fig, plotType):
    """
    Saves a figure to corresponding directory.
    """
    # --- Define output subdirectory and file path ---
    outputFilePath = GetOutputFile(ModelData, outputPlottingDirectory)
    outputFile = os.path.join(outputFilePath,f"{plotType}.png")

    # --- Save figure ---
    fig.savefig(outputFile, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved to {outputFile}")

In [ ]:
####################################
#PLOTTING
fontSettings = {
    "tickFont": 16,
    "labelFont": 10,
    "legendFont": 14,
    "titleFont": 22+4,
}

In [ ]:
varNames = ["refl10cm", 
            "qc+qi+qg","w",
            "qr","relhum"]
variablesDictionary = {}
for varName in tqdm(varNames):
    
    [varData_NSSL,varData_TEMPO, subsetData_NSSL,subsetData_TEMPO, varMean_NSSL,varMean_TEMPO, varProfile_NSSL,varProfile_TEMPO] = LoadOrSave(varName=varName) #GetAllNeededVariables
    variablesDictionary = MakeVariablesDictionary(varName,
                                                  varData_NSSL, varData_TEMPO,
                                                  subsetData_NSSL, subsetData_TEMPO,
                                                  varMean_NSSL, varMean_TEMPO,
                                                  varProfile_NSSL, varProfile_TEMPO,
                                                  variablesDictionary)

In [ ]:
import matplotlib.ticker as ticker

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 8))

# --- Plot 1: NSSL ---
mpType = "NSSL"
p1 = PlotFullReflectivity(mpType, ax1)
cb1 = fig.colorbar(p1, ax=ax1, pad=0.02)

# --- Plot 2: TEMPO ---
mpType = "TEMPO"
p2 = PlotFullReflectivity(mpType, ax2)
# Add colorbar
cb2 = fig.colorbar(p2, ax=ax2, pad=0.02)

SaveFigure(ModelData_NSSL, fig, plotType="ReflectivityBoundingBox")

In [ ]:
varNames = ["refl10cm", "qc+qi+qg","w","qr","relhum"]
for varName in varNames:
    fig = PlotHorizontalVertical_V2(variablesDictionary[varName][f"subsetData_NSSL"],variablesDictionary[varName][f"varMean_NSSL"],
                                    variablesDictionary[varName][f"subsetData_TEMPO"],variablesDictionary[varName][f"varMean_TEMPO"],
                                    varName=varName,
                                    cmap=cmapDictionary[varName],levels=climLevelsDictionary[varName])
    SaveFigure(ModelData_NSSL, fig, plotType=f"HorizontalVertical_{varName}")

In [ ]:
#Plotting Profiles

# 1. Setup the figure and GridSpec
fig = plt.figure(figsize=(22, 6))
# wspace=0.02 makes the plots sit very close to each other
gs = gridspec.GridSpec(1, 5, figure=fig, wspace=0.02)

# Create the axes list using the GridSpec
axs = [fig.add_subplot(gs[i]) for i in range(5)]

# 2. Loop through your variable names and axes
for i, (ax, varName) in enumerate(zip(axs, varNames)):
    
    PlotProfiles(
        variablesDictionary[varName]["varProfile_NSSL"],
        variablesDictionary[varName]["varProfile_TEMPO"], 
        varName,
        ax
    )

    # 3. Strip y-axis for all but the first plot
    if i > 0:
        ax.set_ylabel("")              # Remove text label
        ax.set_yticklabels([])         # Remove tick numbers
        ax.tick_params(axis='y', 
                       which='both', 
                       left=False,     # Remove the dash marks
                       right=False)

# 4. Add the Custom Legend to the middle subplot
custom_lines = [
    Line2D([0], [0], color='blue', lw=2),
    Line2D([0], [0], color='green', lw=2)
]

axs[2].legend(custom_lines, ['NSSL', 'TEMPO'], 
              loc='upper right', 
              fontsize=fontSettings["tickFont"],
              frameon=True)

#Main Title
fig.suptitle(
f"{ModelData_NSSL.region} {ModelData_NSSL.case}",
fontsize=fontSettings['titleFont'],
y=0.95,
fontweight="bold")

SaveFigure(ModelData_NSSL, fig, plotType="VerticalProfiles")

In [ ]:
############################################################################
""" LARGE SUBPLOT """

In [ ]:
# #Testing adding colorbars separately

# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10,4))

# varName="qr"

# # First variable
# p1 = PlotVertical(variablesDictionary[varName][f"varMean_{mpType}"],varName,
#     ax=ax1,cmap=cmapDictionary[varName],levels=climLevelsDictionary[varName])

# fig.colorbar(p1, ax=ax1, pad=0)

# # Second variable
# varName="relhum"
# p2 = PlotVertical(variablesDictionary[varName][f"varMean_{mpType}"],varName,
#                   ax=ax2,cmap=cmapDictionary[varName],levels=climLevelsDictionary[varName])

# fig.colorbar(p2, ax=ax2, pad=0)

# ax1.set_title("qr")
# ax2.set_title("relhum")

In [ ]:
def FormatAllColorbars(fig, integer_axes_list, default_fmt='%.1f', int_fmt='%.0f'):
    """
    Standardizes all colorbars in a figure.
    - integer_axes_list: List of axes whose colorbars should have 0 decimals.
    - default_fmt: Format for all other colorbars (default 1 decimal).
    - int_fmt: Format for specified axes (default 0 decimals).
    """
    for ax in fig.get_axes():
        if ax.get_label() == '<colorbar>':
            # Check if the colorbar belongs to one of our special axes
            try:
                parent_ax = ax._colorbar_info['parents'][0]
                
                if parent_ax in integer_axes_list:
                    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter(int_fmt))
                else:
                    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter(default_fmt))
                
                ax.tick_params(labelsize=fontSettings["labelFont"])
            except (AttributeError, IndexError):
                # Fallback if the internal matplotlib structure differs
                continue
import matplotlib.ticker as ticker

In [ ]:
#Setting Up Figure
fig = plt.figure(figsize=(12,6))
gs = gridspec.GridSpec(3, 4, figure=fig, wspace=0.12, hspace=0.15)
axs = gs.subplots()

########################
mpType = "NSSL"
ax=axs[1,1]
p=PlotFullReflectivity(mpType, ax); fig.colorbar(p, ax=ax, pad=0)
########################
varName="refl10cm"
ax1=axs[0,0]
p1 = PlotVertical(variablesDictionary[varName][f"varMean_{mpType}"],varName,
                  ax=ax1,cmap=cmapDictionary[varName],levels=climLevelsDictionary[varName])
fig.colorbar(p1, ax=ax1, pad=0)
varName="qc+qi+qg"
ax2=axs[0,1]
p2 = PlotVertical(variablesDictionary[varName][f"varMean_{mpType}"],varName,
                  ax=ax2,cmap=cmapDictionary[varName],levels=climLevelsDictionary[varName])
fig.colorbar(p2, ax=ax2, pad=0)

varName="qr"
ax1=axs[2,0]
p1 = PlotVertical(variablesDictionary[varName][f"varMean_{mpType}"],varName,
                  ax=ax1,cmap=cmapDictionary[varName],levels=climLevelsDictionary[varName])
fig.colorbar(p1, ax=ax1, pad=0)
varName="relhum"
ax2=axs[2,1]
p2 = PlotVertical(variablesDictionary[varName][f"varMean_{mpType}"],varName,
                  ax=ax2,cmap=cmapDictionary[varName],levels=climLevelsDictionary[varName])
fig.colorbar(p2, ax=ax2, pad=0)

varName="w"
ax3=axs[1,0]
p3 = PlotVertical(variablesDictionary[varName][f"varMean_{mpType}"],varName,
                  ax=ax3,cmap=cmapDictionary[varName],levels=climLevelsDictionary[varName])
fig.colorbar(p3, ax=ax3, pad=0)


########################
mpType = "TEMPO"
ax=axs[1,2]
p = PlotFullReflectivity(mpType, ax); fig.colorbar(p, ax=ax, pad=0)
########################
varName="refl10cm"
ax1=axs[0,3]
p1 = PlotVertical(variablesDictionary[varName][f"varMean_{mpType}"],varName,
                  ax=ax1,cmap=cmapDictionary[varName],levels=climLevelsDictionary[varName])
fig.colorbar(p1, ax=ax1, pad=0)
varName="qc+qi+qg"
ax2=axs[0,2]
p2 = PlotVertical(variablesDictionary[varName][f"varMean_{mpType}"],varName,
                  ax=ax2,cmap=cmapDictionary[varName],levels=climLevelsDictionary[varName])
cb = fig.colorbar(p2, ax=ax2, pad=0)
# cb.ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))

varName="qr"
ax1=axs[2,3]
p1 = PlotVertical(variablesDictionary[varName][f"varMean_{mpType}"],varName,
                  ax=ax1,cmap=cmapDictionary[varName],levels=climLevelsDictionary[varName])
fig.colorbar(p1, ax=ax1, pad=0)
# cb.ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))
varName="relhum"
ax2=axs[2,2]
p2 = PlotVertical(variablesDictionary[varName][f"varMean_{mpType}"],varName,
                  ax=ax2,cmap=cmapDictionary[varName],levels=climLevelsDictionary[varName])
cb = fig.colorbar(p2, ax=ax2, pad=0)
# cb.ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))

varName="w"
ax3=axs[1,3]
p3 = PlotVertical(variablesDictionary[varName][f"varMean_{mpType}"],varName,
                  ax=ax3,cmap=cmapDictionary[varName],levels=climLevelsDictionary[varName])
cb = fig.colorbar(p3, ax=ax3, pad=0)
# cb.ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))

#fixing up ticks and labels
one=axs[0,1:]; two=axs[2,1:]; three=[axs[1,3]]
for ax in np.concatenate((one, two, three)):
    ax.tick_params(left=False, labelleft=False)
    ax.set_ylabel("")
# for ax in axs.flat:
#     ax.set_title("")
for ax in axs[:-1, :].flat:
    ax.tick_params(bottom=False, labelbottom=False)
    ax.set_xlabel("")
for ax in axs[-1, :].flat:
    ax.set_xlabel("longitude")
axs[1, 1].set_ylabel('latitude')
for ax in [axs[1,1],axs[1,2]]:
    ax.set_ylabel('')
    ax.tick_params(left=False, labelleft=False)

for ax in [axs[0,0], axs[1,0], axs[2,0]]:
    ax.set_ylabel("Altitude (km)")

FormatAllColorbars(fig, [axs[1, 1], axs[1, 2]])

#Main Title
# fig.suptitle(
# f"{ModelData_NSSL.region} {ModelData_NSSL.case}",
# fontsize=fontSettings['titleFont'],
# y=0.95,
# fontweight="bold")

#Saving Figure
SaveFigure(ModelData_NSSL, fig, plotType="CombinedSubplot")

In [ ]:
############################################################################
""" Combining Vertical Profiles acrosss runs """

In [ ]:
def GetFigureFilePath(region,case,spinup_hours,
                      fileName=f"RadarAreaAverages_TZ_T_Combined_NSSLvsMRMSvsTEMPO",
                      extension="png"):
    # --- Define output subdirectory ---
    inputSubDirectory = f"{region}_{case}_{spinup_hours}hrs"
    load_dir = os.path.join(outputPlottingDirectory, inputSubDirectory)
    # --- File path ---
    inputFilePath = os.path.join(
        load_dir,
        f"{fileName}.{extension}"
    )
    return inputFilePath

def GetFilePaths(fileName):
    caseList = ConsolidateFigures_CLASS.GetCaseList()
    filePaths = []
    for region, case, spinup_hours in caseList:
        filePaths.append(GetFigureFilePath(region,case,spinup_hours,
                                           fileName))
    return filePaths

In [ ]:

sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_Plotting import ConsolidateFigures_CLASS

In [ ]:

filePaths = GetFilePaths(fileName=f"VerticalProfiles")

fig = ConsolidateFigures_CLASS.AssembleImageGrid(filePaths=filePaths,
                                                 nrows=6,ncols=1,
                                                 figsize=(6, 6),
                                                 wspace=0.005,hspace=0.005,
                                                 dpi=300)
ConsolidateFigures_CLASS.SaveCombinedFigure(fig, saveDirectory=outputPlottingDirectory,fileName=dataType+"_CombinedProfiles", dpi=900)

In [ ]:
############################################################################
""" OTHER """

In [ ]:
############################################################################
""" ROTATED RECTANGLE """

In [ ]:
####################################
#CALCULATION FUNCTIONS

In [ ]:
def InterpToXYPlane(data,
                    lon0, lat0,
                    lon1, lat1,
                    width,
                    dx=0.01,
                    dy=0.01):

    # direction vector
    dx_line = lon1 - lon0
    dy_line = lat1 - lat0
    L = np.sqrt(dx_line**2 + dy_line**2)

    ux = dx_line / L
    uy = dy_line / L

    # perpendicular vector
    px = uy
    py = -ux

    # midpoint of center line
    lonc = (lon0 + lon1) / 2
    latc = (lat0 + lat1) / 2

    # determine number of grid points
    nx = max(2, int(np.ceil(L / dx)))
    ny = max(2, int(np.ceil(width / dy)))

    # coordinates in rotated frame
    x = np.linspace(-L/2, L/2, nx)
    y = np.linspace(-width/2, width/2, ny)

    X, Y = np.meshgrid(x, y)

    # transform to lon/lat
    lon_plane = lonc + ux*X + px*Y
    lat_plane = latc + uy*X + py*Y

    # interpolate
    interpData = data.interp(
        longitude=(("y","x"), lon_plane),
        latitude=(("y","x"), lat_plane)
    )

    interpData = interpData.assign_coords(
        x=("x", x),
        y=("y", y)
    )
    interpData = interpData.isel(x=slice(None, None, -1))
    interpData = interpData.assign_coords(x=interpData.x[::-1])

    return interpData

In [ ]:
####################################
#PLOTTING FUNCTIONS

In [ ]:
def PlotRotatedRectangleSlice(subsetData, sliceXY,
                              varName,
                              lon0, lat0,
                              lon1, lat1,
                              width):

    fig = plt.figure(figsize=(12,4))

    # outer grid
    outer = gridspec.GridSpec(1, 2, width_ratios=[20,1], wspace=0.05)

    # inner grid for the two plots
    inner = gridspec.GridSpecFromSubplotSpec(
        1, 2, subplot_spec=outer[0], wspace=0.25
    )

    ax0 = fig.add_subplot(inner[0])
    ax1 = fig.add_subplot(inner[1])
    cax = fig.add_subplot(outer[1])

    # --- horizontal map ---
    p1 = subsetData.sel(nVertLevels=3000, method='nearest').plot(
        ax=ax0,
        cmap='turbo',
        add_colorbar=False
    )

    # draw rectangle
    PlotRectangleFromCenterLine(
        ax0,
        lon0, lat0,
        lon1, lat1,
        width
    )

    # --- rotated slice ---
    p2 = sliceXY.sel(nVertLevels=3000, method='nearest').plot(
        ax=ax1,
        cmap='turbo',
        add_colorbar=False
    )

    ax0.set_title("Horizontal Zoom-In")
    ax1.set_title("Rotated Slice")

    # shared colorbar
    fig.colorbar(p1, cax=cax, label=varName)

    plt.show()

In [ ]:
####################################
#PLOTTING

In [ ]:
lon0=-92.25; lat0=29.6
lon1=-94.25; lat1=28.8
width=0.75

sliceXY_TEMPO = InterpToXYPlane(
        subsetData_TEMPO,
        lon0, lat0,
        lon1, lat1,
        width
    )

PlotRotatedRectangleSlice(
    subsetData_TEMPO,
    sliceXY_TEMPO,
    "Reflectivity (dBZ)",
    lon0, lat0,
    lon1, lat1,
    width
)

In [ ]:
sliceXY_TEMPO.mean(dim='y').plot(cmap='turbo')

In [ ]:
lon0=-92.25; lat0=29.6
lon1=-94.25; lat1=28.8
width=0.75

sliceXY_NSSL = InterpToXYPlane(
        subsetData_NSSL,
        lon0, lat0,
        lon1, lat1,
        width
    )

PlotRotatedRectangleSlice(
    subsetData_NSSL,
    sliceXY_NSSL,
    "Reflectivity (dBZ)",
    lon0, lat0,
    lon1, lat1,
    width
)

In [ ]:
sliceXY_NSSL.mean(dim='y').plot(cmap='turbo')

In [ ]:
############################################################################
""" ConvertLonLatToCartesian """

In [ ]:
def ConvertLonLatToCartesian(data):

    lon = data.longitude
    lat = data.latitude

    lon0 = float(lon.min())
    lat0 = float(lat.min())

    # convert degrees → km
    x = (lon - lon0) * 111 * np.cos(np.deg2rad(float(lat.mean())))
    y = (lat - lat0) * 111

    # assign coordinates
    data_xy = data.assign_coords(
        x=("longitude", x.data),
        y=("latitude", y.data)
    )

    # replace dimensions
    data_xy = data_xy.swap_dims({
        "longitude": "x",
        "latitude": "y"
    })

    return data_xy

In [ ]:
#Threshold Profile
# varMean_NSSL.where(varMean_NSSL>0.1).mean(dim='longitude').plot(color='blue',y='nVertLevelsP1')
# varMean_TEMPO.where(varMean_TEMPO>0.1).mean(dim='longitude').plot(color='green',y='nVertLevelsP1')

# varMean_NSSL.where(varMean_NSSL<-0.1).mean(dim='longitude').plot(color='blue',y='nVertLevelsP1')
# varMean_TEMPO.where(varMean_TEMPO<-0.1).mean(dim='longitude').plot(color='green',y='nVertLevelsP1')
# plt.ylim(0,16000)